# SHAP Composite Analysis – Exploratory Notebook

Interactive companion to `scripts/shap_composite.py`.

For each variable the **composite** is:
```
anomaly = event_mean_field − day-of-year climatology (1980–2010)
```
The SHAP mean-sum array is overlaid as a semi-transparent `contourf`.

**Prerequisites:** run `shap_regression.py --meansum` first.

In [ ]:
from __future__ import annotations
import glob, json, os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.basemap import Basemap
%matplotlib inline
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# reuse helpers from the production script
import sys; sys.path.insert(0, '.')
from scripts.shap_composite import (
    _load_configs, _open_dataset, _compute_climatology,
    _event_anomaly_mean, _load_shap, _shap_path,
    _global_max, _combined_max, _normalise,
    _build_cmaps, _make_basemap, _composite_figure,
    VAR_META, CLIM_START, CLIM_END,
)

## 1. Configuration

In [ ]:
MODEL_CONFIG      = './config/model.json'
CASE_STUDIES_FILE = './config/case_studies.json'
SHAP_DIR          = './shap/mod512'

domain, model_name, hw_cases, nohw_cases = _load_configs(
    MODEL_CONFIG, CASE_STUDIES_FILE
)
with open(MODEL_CONFIG) as f:
    cfg = json.load(f)

dataset_path = cfg['datasets']['prs_dataset']

lon_range = np.arange(
    domain['longitude_min'],
    domain['longitude_max'] + domain['resolution'],
    domain['resolution'],
)
lat_range = np.arange(
    domain['latitude_min'],
    domain['latitude_max'] + domain['resolution'],
    domain['resolution'],
)
X, Y = np.meshgrid(lon_range, lat_range)
print(f'Domain  lon {lon_range[0]}..{lon_range[-1]}'
      f'  lat {lat_range[0]}..{lat_range[-1]}')
print(f'{len(hw_cases)} HW  +  {len(nohw_cases)} NO-HW case studies')

## 2. Open Dataset & Compute Climatology

In [ ]:
ds          = _open_dataset(dataset_path, domain)
climatology = _compute_climatology(ds)
print('Dataset variables:', list(ds.data_vars))
print(f'Climatology computed  ({CLIM_START} – {CLIM_END})')

## 3. Select Variable & Case Study

In [ ]:
VAR        = 'msl'   # z500 | msl | peva | sm
EVENT_TYPE = 'hw'    # hw   | nohw
CS_IDX     = 0       # 0-based index in the case-study list

cases = hw_cases if EVENT_TYPE == 'hw' else nohw_cases
case  = cases[CS_IDX]
print(f"Case : {case['label']}  ({case['start']} → {case['end']})")

## 4. Composite Anomaly

In [ ]:
meta      = VAR_META[VAR]
composite = _event_anomaly_mean(
    ds, case['start'], case['end'], climatology, meta['field']
)
print(f'Composite shape: {composite.shape}')
print(f'  min={composite.min():.4g}  mean={composite.mean():.4g}  max={composite.max():.4g}')

fig, ax = plt.subplots(figsize=(11, 6))
m = _make_basemap(ax, domain)
cmap_comp, _ = _build_cmaps(VAR)
vmin = meta['vmin'] or float(composite.min())
vmax = meta['vmax'] or float(composite.max())
img = m.imshow(composite, cmap=cmap_comp, vmin=vmin, vmax=vmax)
m.colorbar(img, extend='both', location='bottom', pad=0.3)
ax.set_title(f"Composite anomaly – {VAR.upper()} – {case['label']}")
plt.tight_layout(); plt.show()

## 5. Load & Normalise SHAP Values

In [ ]:
cs_ids = [c['id'] for c in cases]
norm   = (_global_max(SHAP_DIR, EVENT_TYPE, cs_ids, VAR, model_name)
          if VAR in ('z500', 'msl')
          else _combined_max(SHAP_DIR, cs_ids, VAR, model_name))
print(f'Normalisation constant: {norm:.6g}')

shap_raw  = _load_shap(_shap_path(SHAP_DIR, EVENT_TYPE, case['id'], VAR, model_name))
shap_norm = _normalise(shap_raw, norm, meta['threshold'])
print(f'SHAP shape: {shap_norm.shape}')
print(f'  min={shap_norm.min():.4f}  max={shap_norm.max():.4f}')

## 6. Composite + SHAP Overlay

In [ ]:
_, cmap_shap = _build_cmaps(VAR)
cf_band      = meta['cf_band']
N_COLORS     = 21
shap_levels  = np.linspace(-1.0, 1.0, N_COLORS)
comp_levels  = np.linspace(vmin, vmax, N_COLORS)

fig, ax = plt.subplots(figsize=(12, 7))
m = _make_basemap(ax, domain)

# 1. Background composite
img_comp = m.imshow(composite, cmap=cmap_comp, vmin=vmin, vmax=vmax)

# 2. SHAP contourf overlay
if cf_band is not None:
    active_levels = np.concatenate([
        shap_levels[:cf_band], shap_levels[N_COLORS - cf_band:]
    ])
else:
    active_levels = shap_levels

img_shap = ax.contourf(
    X, Y, np.clip(shap_norm, -1.0, 1.0),
    levels=active_levels, cmap=cmap_shap,
    vmin=-1.0, vmax=1.0, alpha=0.85, extend='both',
)

# 3. Isobar lines
ax.contour(X, Y, composite, levels=comp_levels,
           linewidths=0.9, colors='k', alpha=0.35)

# 4. Labels & colourbars
ax.set_title(f'({VAR.upper()})', loc='left',  fontsize=14, fontweight='bold')
ax.set_title(f"{case['label']}", loc='right', fontsize=12)
fig.colorbar(img_comp, ax=ax, extend='both', fraction=0.025, pad=0.01,
             shrink=0.85).set_label(VAR.upper())
fig.colorbar(img_shap, ax=ax, extend='both', fraction=0.025, pad=0.05,
             shrink=0.85).set_label('Norm. SHAP')
fig.tight_layout(); plt.show()

## 7. All Case Studies at Once

Loop over all case studies for the selected variable and event type,
display inline, then save to disk.

In [ ]:
OUT_DIR = f'./shap/comp_shap/{VAR}'
os.makedirs(OUT_DIR, exist_ok=True)

for case in cases:
    comp_i    = _event_anomaly_mean(
        ds, case['start'], case['end'], climatology, meta['field'])
    shap_i    = _normalise(
        _load_shap(_shap_path(SHAP_DIR, EVENT_TYPE, case['id'], VAR, model_name)),
        norm, meta['threshold'])

    vmin_i = meta['vmin'] or float(comp_i.min())
    vmax_i = meta['vmax'] or float(comp_i.max())

    _composite_figure(
        comp_i, shap_i, domain, X, Y, lon_range, lat_range,
        cmap_comp, cmap_shap,
        var=VAR,
        title=f"{'HW' if EVENT_TYPE=='hw' else 'NO-HW'} {case['id']} – {case['label']}",
        out_path=os.path.join(
            OUT_DIR,
            f"comp_shap_{EVENT_TYPE}{case['id']}_{VAR}{model_name}.png"
        ),
        vmin=vmin_i, vmax=vmax_i, cf_band=meta['cf_band'],
    )
    print(f"Saved: CS {case['id']} – {case['label']}")

## 8. Full Batch Run

To generate all figures for all variables at once:

```bash
python scripts/shap_composite.py \
    --config config/model.json \
    --case-studies config/case_studies.json \
    --vars z500 msl peva sm \
    --verbose
```

Figures are saved under `shap/comp_shap/{var}/`.